# Token Buffer Memory

> **Trim conversation history to fit a strict token budget. Use a real tokenizer, because message count is a poor proxy for context window usage.**

Think of packing a suitcase with a strict weight limit. You don't count the number of items. You weigh each one and remove the oldest items until everything fits. Token Buffer Memory works the same way. It measures conversation history by tokens (not message count) and removes the oldest messages until the total fits under a budget.

Sliding window memory (technique 02) drops messages based on a fixed count. But one message with a code block might cost thousands of tokens, while a short "yes" costs one. Token Buffer Memory replaces that blunt count with precise measurement. A **tokenizer** is a tool that splits text into **tokens** (the word-pieces a model processes internally). Integrate a library like `tiktoken`. Now the system knows exactly how many tokens the history consumes before it reaches the model.

This precision matters most in production. An oversized prompt either truncates silently or throws a hard error. An undersized prompt wastes expensive context window space. Token Buffer Memory eliminates both failure modes. It sets a hard ceiling (`max_token_limit`) and evicts the oldest messages when the total exceeds it.

The technique is deliberately not fancy: no summarization, no embeddings (vector representations of text), no retrieval. That simplicity is its strength. It's deterministic, fast, and straightforward to reason about. When you need more sophisticated retention, layer this with summary or retrieval-augmented memory. Token Buffer Memory remains the foundational guardrail that keeps the prompt within bounds.

**By the end you'll understand:**
- How tokenizers measure real context usage and why token counts beat message counting.
- How to build a token-aware memory from scratch with `tiktoken` and the OpenAI SDK.
- When token budgets shine and when you need something more.

## Key Concepts

- **Token**: The smallest unit a language model reads. Roughly one word or word-piece. The word "unhappiness" might split into two tokens: "un" and "happiness".
- **Tokenizer (`tiktoken`)**: A library that splits text into tokens for a specific model. OpenAI's `tiktoken` handles GPT-4, GPT-4o, and related models.
- **Encoding**: The specific token vocabulary a model uses. GPT-4 uses `cl100k_base`. GPT-4o uses `o200k_base`. The wrong encoding gives inaccurate counts.
- **max_token_limit**: A hard ceiling on total tokens allowed in conversation history. The system evicts old messages until the total stays at or below this number.
- **Per-message overhead**: Each chat message carries formatting tokens beyond the content itself. These include role markers, a content separator, and end-of-message delimiters. Typically 4 tokens per message.
- **Message-level eviction**: Entire messages are removed (not partially trimmed). This preserves coherence but may slightly under-use the budget.
- **Context window**: The maximum number of tokens a model can process in one call. GPT-4o supports 128K tokens. Your history budget must leave room for the system prompt and the model's response.

## Architecture

<p align="center">
  <img src="../../images/diagrams/05_token_buffer_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    NewMsg["New Message"] --> Tokenizer["Tokenizer\n(tiktoken)"]
    Tokenizer --> Counter["Token Counter\n(sum all messages)"]
    Counter --> BudgetCheck{"Total > \nmax_token_limit?"}
    BudgetCheck -- No --> TrimmedHistory["Trimmed History"]
    BudgetCheck -- Yes --> Evict["Evict Oldest\nMessage"]
    Evict --> Counter
    TrimmedHistory --> LLM["LLM"]
    LLM --> Response["Response"]
    Response --> NewMsg
```

</details>

**Data flow:** Each new message is tokenized. The counter sums all messages in the history. If the total exceeds `max_token_limit`, the oldest message is evicted and the count rechecks in a loop. Once under budget, the trimmed history goes to the LLM. The response is appended and the cycle repeats.

## Setup

Install dependencies and configure API access.

In [ ]:
%pip install -q tiktoken openai python-dotenv

Import `tiktoken` for token counting and the OpenAI SDK for LLM calls. The API key loads from a `.env` file.

In [ ]:
import os

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env

client = OpenAI()
MODEL = "gpt-4o"
ENCODING = tiktoken.encoding_for_model(MODEL)

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

## Implementation

We'll build a `TokenBufferMemory` class that:
1. Counts tokens per message using `tiktoken`.
2. Keeps a running total of all stored messages.
3. Evicts the oldest messages when the total exceeds `max_token_limit`.
4. Sends the trimmed history to the OpenAI API on each call.

### Token Counting

Accurate token counting is the foundation of this technique. Each chat message costs more tokens than its content alone. The Chat Completions API wraps every message with special tokens. These include role markers, a content separator, and end-of-message delimiters. This overhead is typically 4 tokens per message.

Let's see how `tiktoken` counts tokens for messages of different lengths.

In [ ]:
def count_message_tokens(message: dict) -> int:
    """Count tokens for one chat message, including per-message overhead.

    Overhead (4 tokens per message):
      <|im_start|> + role + content separator + <|im_end|>
    """
    overhead = 4
    return overhead + len(ENCODING.encode(message["content"]))


# Demo: count tokens for messages of different lengths
sample_messages = [
    {"role": "user", "content": "yes"},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "user", "content": (
        "Write a Python function that computes the nth Fibonacci number "
        "using dynamic programming with memoization."
    )},
]

for msg in sample_messages:
    tokens = count_message_tokens(msg)
    print(f"  {tokens:3d} tokens | \"{msg['content'][:70]}\"")

### The TokenBufferMemory Class

This class wraps token counting into a complete memory system. On each `chat()` call, it:
1. Appends the user message.
2. Sums tokens across all stored messages.
3. Evicts the oldest messages in a loop until the total fits under `max_token_limit`.
4. Sends the trimmed history to the LLM.
5. Appends the response.

Set `max_token_limit` to the space available for history. Subtract the system prompt tokens and your response budget from the model's context window. For example, with a 128K window, a 200-token system prompt, and a 1,024-token response budget, set `max_token_limit` to about 126,000.

In [ ]:
class TokenBufferMemory:
    """Token-aware conversation memory with a strict token budget."""

    def __init__(
        self,
        model: str = "gpt-4o",
        max_token_limit: int = 2048,
        system_prompt: str | None = None,
        max_response_tokens: int = 512,
    ):
        self.client = OpenAI()
        self.model = model
        self.max_token_limit = max_token_limit
        self.system_prompt = system_prompt
        self.max_response_tokens = max_response_tokens

        # Select the correct encoding for the target model
        self.encoding = tiktoken.encoding_for_model(model)

        # Core data structure: the message buffer
        self.messages: list[dict] = []

        # Track evictions for inspection
        self.eviction_log: list[dict] = []

    # -- Token counting -----------------------------------------------
    def count_tokens(self, message: dict) -> int:
        """Count tokens for one message including per-message overhead."""
        overhead = 4  # <|im_start|>, role, separator, <|im_end|>
        return overhead + len(self.encoding.encode(message["content"]))

    def total_tokens(self) -> int:
        """Sum token counts across all stored messages."""
        return sum(self.count_tokens(m) for m in self.messages)



Now we add the eviction loop and the `chat` method. The `_trim` method runs a while-loop: it pops the oldest message, subtracts its token count, and repeats until the total fits under `max_token_limit`. The `chat` method appends the user message, trims if needed, calls the LLM, and appends the response.

In [ ]:
    # -- Eviction loop ------------------------------------------------
    def _trim(self) -> None:
        """Remove oldest messages until total tokens fit within budget."""
        total = self.total_tokens()
        while total > self.max_token_limit and len(self.messages) > 1:
            removed = self.messages.pop(0)
            removed_tokens = self.count_tokens(removed)
            total -= removed_tokens
            self.eviction_log.append({
                "role": removed["role"],
                "preview": removed["content"][:60],
                "tokens_freed": removed_tokens,
            })

    # -- Chat ---------------------------------------------------------
    def chat(self, user_input: str) -> str:
        """Send a message, trim if over budget, return the response."""
        self.messages.append({"role": "user", "content": user_input})
        self._trim()

        # Build the API payload
        api_messages = []
        if self.system_prompt:
            api_messages.append({"role": "system", "content": self.system_prompt})
        api_messages.extend(self.messages)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=api_messages,
            max_tokens=self.max_response_tokens,
        )

        assistant_text = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_text})

        return assistant_text



Utility methods for inspecting and resetting the memory. `get_messages` returns a copy of the current history. `clear` wipes both messages and the eviction log.

In [ ]:
    # -- Utilities ----------------------------------------------------
    def get_messages(self) -> list[dict]:
        """Return a copy of the message history."""
        return [m.copy() for m in self.messages]

    def clear(self) -> None:
        """Reset all messages and logs."""
        self.messages.clear()
        self.eviction_log.clear()

    def __len__(self) -> int:
        return len(self.messages)

    def __repr__(self) -> str:
        return (
            f"TokenBufferMemory("
            f"{len(self.messages)} msgs, "
            f"{self.total_tokens()}/{self.max_token_limit} tokens)"
        )

## Example Run

A short conversation shows token buffer memory in action. With a generous budget (2,048 tokens), all messages fit and nothing gets evicted. The agent remembers everything.

In [ ]:
memory = TokenBufferMemory(
    model="gpt-4o",
    max_token_limit=2048,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer.",
    "I'm building a project about agent memory systems.",
    "What's my name and what am I working on?",  # recall test
]

for msg in exchanges:
    print(f"User:  {msg}")
    reply = memory.chat(msg)
    print(f"Agent: {reply}")
    print(f"       [{memory.total_tokens()} / {memory.max_token_limit} tokens, "
          f"{len(memory)} messages]\n")

### Eviction in Action

Now we'll use a tiny budget (300 tokens). This forces eviction after a few turns. Watch the token count stay bounded as older messages disappear.

In [ ]:
small_memory = TokenBufferMemory(
    model="gpt-4o",
    max_token_limit=300,  # deliberately small to trigger eviction
    system_prompt="Reply in one short sentence.",
)

demo_messages = [
    "My name is Bob and I live in Tokyo.",
    "I have two cats named Mochi and Sushi.",
    "My favorite language is Rust.",
    "I work at a robotics startup with 20 engineers.",
    "What do you remember about me?",  # some early facts will be gone
]

for msg in demo_messages:
    tokens_before = small_memory.total_tokens()
    reply = small_memory.chat(msg)
    tokens_after = small_memory.total_tokens()
    print(f"User:  {msg}")
    print(f"Agent: {reply}")
    print(f"       [tokens: {tokens_before} -> {tokens_after}, "
          f"messages: {len(small_memory)}]\n")

# Show what got evicted
if small_memory.eviction_log:
    print("--- Eviction log ---")
    for entry in small_memory.eviction_log:
        print(f"  Removed ({entry['role']}): "
              f"\"{entry['preview']}\" "
              f"({entry['tokens_freed']} tokens freed)")

### Why Tokens Beat Message Count

A sliding window that keeps the last 5 messages treats all messages as equal. In practice, messages vary in size by orders of magnitude. A one-word reply and a pasted code block both count as "one message" in a sliding window. But they differ enormously in tokens.

The cell below shows this directly.

In [ ]:
comparison_messages = [
    {"role": "user", "content": "yes"},
    {"role": "assistant", "content": "No problem!"},
    {"role": "user", "content": (
        "Here's my entire configuration file:\n"
        "import logging\n"
        "import os\n"
        "from pathlib import Path\n\n"
        "BASE_DIR = Path(__file__).resolve().parent\n"
        "SECRET_KEY = os.environ['SECRET_KEY']\n"
        "DEBUG = os.environ.get('DEBUG', 'False') == 'True'\n"
        "ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', '').split(',')\n"
        "DATABASES = {\n"
        "    'default': {\n"
        "        'ENGINE': 'django.db.backends.postgresql',\n"
        "        'NAME': os.environ['DB_NAME'],\n"
        "        'USER': os.environ['DB_USER'],\n"
        "        'PASSWORD': os.environ['DB_PASSWORD'],\n"
        "        'HOST': os.environ.get('DB_HOST', 'localhost'),\n"
        "        'PORT': os.environ.get('DB_PORT', '5432'),\n"
        "    }\n"
        "}\n"
        "LOGGING = {\n"
        "    'version': 1,\n"
        "    'handlers': {\n"
        "        'console': {'class': 'logging.StreamHandler'},\n"
        "    },\n"
        "    'root': {'handlers': ['console'], 'level': 'WARNING'},\n"
        "}\n"
    )},
]

print("Message count vs. token count:\n")
total_tokens = 0
for i, msg in enumerate(comparison_messages):
    tokens = count_message_tokens(msg)
    total_tokens += tokens
    preview = msg["content"][:50].replace("\n", " ")
    suffix = "..." if len(msg["content"]) > 50 else ""
    print(f"  Message {i + 1} ({msg['role']:>9}): {tokens:4d} tokens "
          f"| \"{preview}{suffix}\"")

print(f"\n  Total: {total_tokens} tokens across {len(comparison_messages)} messages")
print(f"\n  A 3-message sliding window keeps all three.")
print(f"  A 50-token budget would drop message 3 "
      f"({count_message_tokens(comparison_messages[2])} tokens).")
print(f"  Token-based trimming makes the right call here.")

## Tradeoffs

### When Token Buffer Memory Works Well
- **Precise budget control**: You know exactly how many tokens go to the model. No silent truncation, no rejected prompts.
- **Variable message sizes**: Code blocks, JSON payloads, and short replies all get measured accurately. Message-count trimming can't do this.
- **Speed**: Eviction runs in milliseconds with no LLM calls (unlike summary memory). The trimming is deterministic.
- **Multi-model routing**: If your system routes to models with different context windows, set a per-model `max_token_limit`. The same class handles both.

### When It Breaks Down
- **No recall of evicted content**: Once a message is removed, that information is gone. Summary memory or retrieval-augmented memory preserves some of it.
- **One large message crowds out many small ones**: A single 1,000-token message forces eviction of many short messages to make room.
- **Tokenizer must match the model**: Using GPT-4's encoding with GPT-4o (or vice versa) gives wrong counts. Keep the encoding in sync with the model.
- **Whole-message eviction can waste budget**: If the budget is 500 tokens and the oldest message costs 400, evicting it frees more space than needed. Partial trimming would be more efficient, but it breaks message coherence.

### What's Next?
- **[06: Vector-Store Memory](../06_vector_store_memory/)** retrieves relevant past messages using semantic search instead of dropping by age.
- **[03: Summary Memory](../03_summary_memory/)** compresses old messages into a summary so no information is fully lost.
- Combine approaches: use token buffer as the hard ceiling, with summary or retrieval to preserve evicted content.

## Further Reading

- [LangChain ConversationTokenBufferMemory](https://python.langchain.com/docs/modules/memory/types/token_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): LangChain's built-in implementation. It wraps tiktoken for automatic token-based trimming.
- [tiktoken: OpenAI's Fast BPE Tokenizer](https://github.com/openai/tiktoken): The reference tokenizer for OpenAI models.
- [OpenAI Cookbook: How to Count Tokens with tiktoken](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Step-by-step guide to counting tokens accurately. Covers per-message overhead for the Chat Completions API.
- [Anthropic Token Counting API](https://docs.anthropic.com/en/docs/build-with-claude/token-counting?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Anthropic's server-side token counting endpoint for Claude models.

---

*← Previous: [04: Summary Buffer Memory](../04_summary_buffer_memory/) · Next: [06: Vector-Store Memory](../06_vector_store_memory/) →*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Priority-based trimming
Modify `_trim()` so it scores each message (e.g., messages containing questions or names get higher priority) and evicts the lowest-priority message first instead of the oldest. Run a 15-turn conversation and compare recall against the default FIFO approach.

### Challenge 2: Budget sweep
Run the same conversation with `max_token_limit` set to 500, 1000, 2000, and 4000. For each budget, record the number of messages retained, the number evicted, and how many recall questions the agent answers correctly. Present results in a table.

### Challenge 3: Token budget with summary overflow
When `_trim()` evicts messages, pass them to a summarizer instead of discarding them. Prepend the summary as a system-prompt addition. Measure total tokens used per turn and compare against pure token buffer. This combines 05 Token Buffer with the approach from 03 Summary Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--05-token-buffer-memory--token-buffer-memory)
